# CT-CBM Colab Notebook

This notebook runs the CT-CBM pipeline. It intentionally excludes the C3M and CB-LLM comparison branches.

The pipeline follows the repository implementation:
1. Load a text-classification dataset and fine-tune the black-box encoder on task labels.
2. Build concept annotations with the `ours` method: extract micro-topics, cluster them, summarize clusters into macro-concepts, and create binary concept columns.
3. Build mean CAVs from the annotated data using the fine-tuned black-box encoder.
4. Compute concept identifiability, LIG importance, combined concept scores, cumulative coverage, and save the ranking artifact expected by CT-CBM.
5. Train the joint CBM and the residual fitting loop used by CT-CBM.
6. Print test classification reports for both CT-CBM simple and CT-CBM residual branches.

In [ ]:
# Colab setup.
# Run this cell once after opening the notebook in Colab.
# If you already installed the dependencies in the runtime, you can skip it.

!pip -q install \
  transformers datasets accelerate sentence-transformers \
  scikit-learn pandas numpy tqdm matplotlib tabulate joblib \
  umap-learn hdbscan captum

In [ ]:
# Repository setup.
# This notebook is designed to be the single Colab control file for the project.
# It still uses the Python modules in run_experiments/ because those modules contain
# the paper implementation of annotation, CAV creation, ranking, and CT-CBM training.
#
# Option A: If this notebook is opened from a cloned repo, leave PROJECT_DIR as-is.
# Option B: In Colab, set REPO_URL to your GitHub repository URL and run this cell.
# Option C: Upload the repository folder/zip to /content and set PROJECT_DIR manually.

import os
from pathlib import Path

REPO_URL = "https://github.com/dauduchieu/iSE-CT-CBM.git"
PROJECT_DIR = Path("/content/CT-CBM")

if not PROJECT_DIR.exists() and REPO_URL:
    !git clone {REPO_URL} {PROJECT_DIR}

if not PROJECT_DIR.exists() and Path.cwd().name == "CT-CBM":
    PROJECT_DIR = Path.cwd()

if not (PROJECT_DIR / "run_experiments").exists():
    raise FileNotFoundError(
        "Cannot find run_experiments/. Set REPO_URL or PROJECT_DIR so Colab can see the repository code."
    )

os.chdir(PROJECT_DIR)
print(f"Project directory: {PROJECT_DIR}")

In [ ]:
# Imports and Python path setup.
# These paths mirror the original notebooks, but are absolute for Colab stability.

import sys, gc, json, pickle, random, warnings, logging
from pathlib import Path

ROOT = Path.cwd()
for rel in ["run_experiments", "run_experiments/scripts", "run_experiments/models", "run_experiments/data"]:
    path = str(ROOT / rel)
    if path not in sys.path:
        sys.path.append(path)

import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")
for logger in [logging.getLogger(name) for name in logging.root.manager.loggerDict]:
    if "transformers" in logger.name.lower():
        logger.setLevel(logging.ERROR)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.device("cuda" if torch.cuda.is_available() else "cpu"))

In [ ]:
# Experiment configuration.
# The default is intentionally small enough to start on Colab.
# Increase epochs, clusters, and row limits for full paper-scale experiments.

MODEL_NAME = "bert-base-uncased"     # Supported: "bert-base-uncased", "deberta-large", "gemma".
DATASET = "agnews"                  # Options: "agnews", "movies", "dbpedia", "medical", "pubmed", "stackoverflow", "medical_2", "ecom", "sst2".
ANNOTATION = "our_annotation"       # CT-CBM in this notebook always uses our annotation.

# Annotation model used by the ours pipeline.
# The original code uses a chat-style causal LM. Gemma requires a Hugging Face token and license access.
DISCOVERY_MODEL_NAME = "google/gemma-2-2b-it"
HF_TOKEN = ""  # In Colab, you can also load this from userdata if preferred.

# Runtime controls.
RUN_ANNOTATION = True                # Set False to reuse existing df_with_topics_v4*.csv files.
RUN_CAVS = True                      # Set False to reuse existing CAV JSON.
RUN_RANKING = True                   # Set False to reuse existing ranking JSON.
RUN_CT_CBM_TRAINING = True           # Set False to prepare all artifacts without fitting CT-CBM.
RUN_BLACKBOX_TRAINING = True          # Fine-tune a black-box BERT classifier before CT-CBM.
RUN_LIG_RANKING = True               # Use LIG importance in the CT-CBM concept ranking, as in the paper notebooks.

# Debug controls. Use small values first to verify the full path; set to None for full data.
MAX_TRAIN_ROWS = 400                 # None for full train set.
MAX_VAL_ROWS = 200                   # None for full validation set.
MAX_TEST_ROWS = 200                  # None for full test set.
N_CLUSTERS = 100                      # Paper-scale runs can use larger values, e.g. 100.

# Black-box fitting controls.
BLACKBOX_NUM_EPOCHS = 5

# LIG ranking controls.
LIG_ATTRIBUTION_BATCH_SIZE = 32

# CT-CBM fitting controls.
NUM_ITERATIONS = 5
NUM_EPOCHS_JOINT = 5
NUM_EPOCHS_RESIDUAL = 5
COVERAGE_THRESHOLD = 0.80
F1_CUTOFF = 0.0

In [ ]:
# Load the repository config and override storage paths for Colab.
# The original config files contain machine-specific paths; Colab writes to /content/ct_cbm_outputs.

from load_config import load_config

config = load_config(MODEL_NAME, DATASET)
config.annotation = ANNOTATION
config.DATASET = DATASET
config.model_name = MODEL_NAME
config.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
config.num_epochs = NUM_EPOCHS_JOINT
config.cavs_type = "mean"
config.cavs_type_arg = "mean"
config.agg_mode = "abs"
config.agg_scope = "all"
config.seed = SEED

OUTPUT_ROOT = Path("/content/ct_cbm_outputs") / DATASET
config.SAVE_PATH = str(OUTPUT_ROOT) + "/"
config.SAVE_PATH_CONCEPTS = str(OUTPUT_ROOT / "concepts_discovery")
Path(config.SAVE_PATH_CONCEPTS).mkdir(parents=True, exist_ok=True)
(Path(config.SAVE_PATH) / "blue_checkpoints" / config.model_name / "cavs" / config.cavs_type).mkdir(parents=True, exist_ok=True)
(Path(config.SAVE_PATH) / "blue_checkpoints" / config.model_name / "Our_CBM_joint").mkdir(parents=True, exist_ok=True)

print("SAVE_PATH:", config.SAVE_PATH)
print("SAVE_PATH_CONCEPTS:", config.SAVE_PATH_CONCEPTS)

In [ ]:
# Load text-classification data.
# The repo data loaders return ordinary and split DataFrames; the annotation pipeline uses the DataFrames.

from prepare_data import load_fc_prepare_data

prepare_data = load_fc_prepare_data(config.DATASET)
train_loader, test_loader, val_loader, train_df, val_df, test_df = prepare_data(config)

# Optional row limits make the first Colab run faster and cheaper.
if MAX_TRAIN_ROWS is not None:
    train_df = train_df.head(MAX_TRAIN_ROWS).copy()
if MAX_VAL_ROWS is not None:
    val_df = val_df.head(MAX_VAL_ROWS).copy()
if MAX_TEST_ROWS is not None:
    test_df = test_df.head(MAX_TEST_ROWS).copy()

# The downstream code expects the two canonical columns: text and label.
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    assert {"text", "label"}.issubset(df.columns), f"{name}_df must contain text and label columns"
    print(name, df.shape, df.head(1).to_dict("records"))

In [ ]:
# Load the text encoder and tokenizer used by the black-box baseline and CT-CBM.
# The first load also returns a simple classifier head for black-box fine-tuning.

from models.utils import load_model_and_tokenizer, RidgeLinearLayer

embedder_model, embedder_tokenizer, ModelXtoCtoY_layer, blackbox_classifier = load_model_and_tokenizer(config, n_concepts=1)

# Compatibility shim for newer Hugging Face versions where some tokenizer classes
# no longer expose encode_plus. The repo dataset calls encode_plus, so alias it
# to the standard tokenizer __call__ API when needed.
try:
    getattr(embedder_tokenizer, "encode_plus")
except AttributeError:
    embedder_tokenizer.encode_plus = embedder_tokenizer.__call__

embedder_model.to(config.device)
print("Loaded embedder:", MODEL_NAME)


In [ ]:
# Fine-tune the black-box BERT baseline.
# This is a standard encoder + linear classifier trained directly on task labels.
# It is useful both as a black-box reference and as the model used by attribution-based steps.

from concepts_bank_utils import create_dataloader

blackbox_train_loader = create_dataloader(train_df[["text", "label"]].copy(), embedder_tokenizer, config.max_len, config.batch_size, shuffle=True)
blackbox_val_loader = create_dataloader(val_df[["text", "label"]].copy(), embedder_tokenizer, config.max_len, config.batch_size, shuffle=False)
blackbox_test_loader = create_dataloader(test_df[["text", "label"]].copy(), embedder_tokenizer, config.max_len, config.batch_size, shuffle=False)

# Temporarily override the epoch count for the black-box baseline only.
_original_num_epochs = config.num_epochs
config.num_epochs = BLACKBOX_NUM_EPOCHS

if config.model_name == "gemma":
    from models.BaselineModel_gemma import BaselineModel
else:
    from models.BaselineModel import BaselineModel

black_box_model = BaselineModel(
    embedder_model=embedder_model,
    classifier=blackbox_classifier,
    train_loader=blackbox_train_loader,
    val_loader=blackbox_val_loader,
    test_loader=blackbox_test_loader,
    config=config,
    save_path=config.SAVE_PATH,
    use_cls_token=config.use_cls_token,
)

if RUN_BLACKBOX_TRAINING:
    black_box_model.train_model()
    # Reload the best validation checkpoint saved by BaselineModel.train_model().
    black_box_model.load_model()
    black_box_model.evaluate_model(blackbox_test_loader, "Test")
    black_box_model.save_performance_json()
else:
    # Reuse an existing fine-tuned black-box checkpoint when training is skipped.
    black_box_model.load_model()
    print("Loaded existing fine-tuned black-box checkpoint.")

config.num_epochs = _original_num_epochs

# Use the fine-tuned black-box encoder as the CT-CBM encoder.
# CT-CBM does not reuse the black-box classifier head; it learns a fresh bottleneck
# layer X->C->Y on top of the fine-tuned encoder.
embedder_model = black_box_model.embedder_model
_tmp_encoder, embedder_tokenizer, ModelXtoCtoY_layer, _ = load_model_and_tokenizer(config, n_concepts=1)
del _tmp_encoder
try:
    getattr(embedder_tokenizer, "encode_plus")
except AttributeError:
    embedder_tokenizer.encode_plus = embedder_tokenizer.__call__
embedder_model.to(config.device)
ModelXtoCtoY_layer.to(config.device)
torch.cuda.empty_cache()
print("Loaded the fine-tuned black-box encoder for CT-CBM.")


In [ ]:
# Load the discovery model for our annotation.
# This model extracts concise micro-topics and summarizes topic clusters into macro-concepts.
# If you reuse existing annotation CSVs, this cell can be skipped by setting RUN_ANNOTATION=False.

from transformers import AutoModelForCausalLM, AutoTokenizer

if RUN_ANNOTATION:
    tokenizer_kwargs = {}
    model_kwargs = {"torch_dtype": torch.float16 if torch.cuda.is_available() else torch.float32}
    if HF_TOKEN:
        tokenizer_kwargs["token"] = HF_TOKEN
        model_kwargs["token"] = HF_TOKEN
    if torch.cuda.is_available():
        model_kwargs["device_map"] = "auto"

    discovery_tokenizer = AutoTokenizer.from_pretrained(DISCOVERY_MODEL_NAME, **tokenizer_kwargs)

    # Compatibility shim for newer tokenizer/transformers combinations.
    # The original repository expects apply_chat_template(..., return_tensors="pt")
    # to return a torch.Tensor, but some Colab versions return tokenizers.Encoding.
    # This wrapper normalizes the output so concepts_bank_utils.py can keep using
    # encoded_input[0][:-2] exactly as in the paper code.
    _raw_apply_chat_template = discovery_tokenizer.apply_chat_template

    def _apply_chat_template_as_tensor(*args, **kwargs):
        result = _raw_apply_chat_template(*args, **kwargs)
        if kwargs.get("return_tensors") != "pt":
            return result
        if isinstance(result, torch.Tensor):
            return result
        if hasattr(result, "input_ids"):
            ids = result.input_ids
            if isinstance(ids, torch.Tensor):
                return ids if ids.dim() == 2 else ids.unsqueeze(0)
            return torch.tensor(ids if isinstance(ids[0], list) else [ids], dtype=torch.long)
        if hasattr(result, "ids"):
            return torch.tensor([result.ids], dtype=torch.long)
        if isinstance(result, list):
            return torch.tensor(result if result and isinstance(result[0], list) else [result], dtype=torch.long)
        raise TypeError(f"Unsupported chat template output type: {type(result)}")

    discovery_tokenizer.apply_chat_template = _apply_chat_template_as_tensor

    discovery_model = AutoModelForCausalLM.from_pretrained(DISCOVERY_MODEL_NAME, **model_kwargs)
    if discovery_tokenizer.pad_token is None:
        discovery_tokenizer.pad_token = discovery_tokenizer.eos_token
    print("Loaded discovery model:", DISCOVERY_MODEL_NAME)
else:
    discovery_model = None
    discovery_tokenizer = None
    print("Skipping discovery model load; existing annotation files will be reused.")

In [ ]:
# Our annotation pipeline.
# This cell creates:
# - df_with_topics_v2.csv: micro-topics extracted from train text.
# - UMAP/HDBSCAN artifacts: reusable train clustering model.
# - df_with_topics_v4.csv: train DataFrame with dummy_<macro concept> columns.
# - df_with_topics_v4_test.csv: test DataFrame projected onto the train macro-concept space.

from annotation_ours import process_and_save_augmentation

if RUN_ANNOTATION:
    process_and_save_augmentation(
        train_df=train_df,
        val_df=val_df,
        test_df=test_df,
        config=config,
        discovery_model=discovery_model,
        discovery_tokenizer=discovery_tokenizer,
        embedder_tokenizer=embedder_tokenizer,
        save_dir=config.SAVE_PATH_CONCEPTS,
        n_cluster=N_CLUSTERS,
    )
else:
    print("Reusing existing annotation artifacts from", config.SAVE_PATH_CONCEPTS)

train_aug_path = Path(config.SAVE_PATH_CONCEPTS) / "df_with_topics_v4.csv"
test_aug_path = Path(config.SAVE_PATH_CONCEPTS) / "df_with_topics_v4_test.csv"
assert train_aug_path.exists(), train_aug_path
assert test_aug_path.exists(), test_aug_path

In [ ]:
# Load and clean our annotated train/test DataFrames.
# Concept columns are stored as dummy_<concept>. The dataloader removes dummy_ when exposing batch keys.

from concepts_bank_utils import clean_concept_name, create_dataloader


def load_our_annotation_frames(config):
    train_aug = pd.read_csv(Path(config.SAVE_PATH_CONCEPTS) / "df_with_topics_v4.csv")
    test_aug = pd.read_csv(Path(config.SAVE_PATH_CONCEPTS) / "df_with_topics_v4_test.csv")

    for df in [train_aug, test_aug]:
        df["text"] = df["text"].astype(str).str.strip()
        df["label"] = df["label"].astype(int)
        for col in [c for c in df.columns if c.startswith("dummy_")]:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

    # Keep only concept columns that exist in both splits so train/test batches have the same keys.
    train_concepts = {c for c in train_aug.columns if c.startswith("dummy_")}
    test_concepts = {c for c in test_aug.columns if c.startswith("dummy_")}
    common_concepts = sorted(train_concepts & test_concepts)
    if not common_concepts:
        raise ValueError("No common dummy_ concept columns found between train and test annotations.")

    base_cols = ["text", "label"]
    train_aug = train_aug[base_cols + common_concepts].copy()
    test_aug = test_aug[base_cols + common_concepts].copy()
    return train_aug, test_aug, common_concepts


df_aug_train, df_aug_test, dummy_concept_cols = load_our_annotation_frames(config)
print("Augmented train:", df_aug_train.shape)
print("Augmented test:", df_aug_test.shape)
print("Number of common concepts:", len(dummy_concept_cols))
print("First concepts:", [c.replace("dummy_", "") for c in dummy_concept_cols[:10]])

In [ ]:
# Create dataloaders for the annotated data.
# The joint CBM trains the concept heads using these binary concept labels.
# The validation loader does not need concept labels in the CT-CBM stopping rule,
# so we recreate it from val_df to honor MAX_VAL_ROWS during Colab debug runs.

train_aug_loader = create_dataloader(df_aug_train, embedder_tokenizer, config.max_len, config.batch_size, shuffle=True)
test_aug_loader = create_dataloader(df_aug_test, embedder_tokenizer, config.max_len, config.batch_size, shuffle=False)
val_loader = create_dataloader(val_df[["text", "label"]].copy(), embedder_tokenizer, config.max_len, config.batch_size, shuffle=False)

print("Train batches:", len(train_aug_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_aug_loader))


In [ ]:
# Build a temporary JointModel wrapper so CAV utilities can reuse get_pooled_output().
# At this stage the CBM bottleneck has one placeholder concept; the real concept count is set later.

from models.jointCBMv2 import JointModel

baseline_joint = JointModel(
    embedder_model,
    embedder_tokenizer,
    ModelXtoCtoY_layer,
    config,
    train_aug_loader,
    val_loader,
)

In [ ]:
# Mean CAV creation.
# For each macro-concept, the CAV is mean(embeddings where concept is present) minus
# mean(embeddings where concept is absent), matching the repo's mean CAV implementation.

from mean_cavs_creation import compute_cavs_mean_minus

cav_json_path = Path(config.SAVE_PATH) / "blue_checkpoints" / config.model_name / "cavs" / config.cavs_type / f"cavs_mean_{config.annotation}.json"

if RUN_CAVS or not cav_json_path.exists():
    cavs = compute_cavs_mean_minus(
        df_aug=df_aug_train.copy(),
        embedder_model=embedder_model,
        embedder_tokenizer=embedder_tokenizer,
        baseline_model=baseline_joint,
        config=config,
    )
else:
    with open(cav_json_path) as f:
        cavs = {k: torch.tensor(v, dtype=torch.float32) for k, v in json.load(f).items()}

print("CAV count:", len(cavs))
print("CAV file:", cav_json_path)

In [ ]:
# Concept detection and filtering.
# This evaluates whether each CAV can recover its annotation column via cosine similarity.
# Concepts with F1 below F1_CUTOFF are dropped from the ranking candidate set.

from new_heuristique import compute_cosine_matrix_and_metrics
from concepts_bank_utils import clean_concept_name as _clean_concept_name

ranking_dir = Path(config.SAVE_PATH) / "blue_checkpoints" / config.model_name / "cavs" / config.cavs_type
ranking_dir.mkdir(parents=True, exist_ok=True)

# Drop stale cosine caches before calling the repository function.
# A stale cache happens when annotation or MAX_TRAIN_ROWS changes between runs.
cosine_cache_path = ranking_dir / f"cosine_df_{config.annotation}.pkl"
if cosine_cache_path.exists():
    try:
        cached_cosine_df = pd.read_pickle(cosine_cache_path)
        expected_cols = {"text", "label", *cavs.keys()}
        cache_ok = (
            cached_cosine_df.index.equals(df_aug_train.index)
            and len(cached_cosine_df) == len(df_aug_train)
            and expected_cols.issubset(set(cached_cosine_df.columns))
        )
        if not cache_ok:
            cosine_cache_path.unlink()
            print("Removed stale cosine cache:", cosine_cache_path)
    except Exception as exc:
        cosine_cache_path.unlink()
        print("Removed unreadable cosine cache:", cosine_cache_path, exc)

if RUN_RANKING:
    cosine_train, cosine_val, cosine_df, thresholds, detection_metrics, filtered_concepts = compute_cosine_matrix_and_metrics(
        df=df_aug_train,
        text_column="text",
        embedder_model=embedder_model,
        embedder_tokenizer=embedder_tokenizer,
        cavs=cavs,
        f1_cutoff=F1_CUTOFF,
        device=config.device,
        save_dir=str(ranking_dir),
        config=config,
        annotation=config.annotation,
        cos_cubed=False,
    )
else:
    metrics_path = ranking_dir / f"detection_concept_{config.cavs_type}_{config.annotation}_{config.agg_mode}_{config.agg_scope}.json"
    with open(metrics_path) as f:
        detection_metrics = json.load(f)
    filtered_concepts = [c for c, m in detection_metrics.items() if m.get("F1", 0.0) >= F1_CUTOFF]

# Rank by detection quality first, then accuracy.
# compute_cosine_matrix_and_metrics returns filtered_concepts after name cleaning,
# while detection_metrics keeps the original CAV keys. Compare with cleaned names
# to avoid accidentally filtering every ranked concept out.
filtered_clean = {_clean_concept_name(c) for c in filtered_concepts}
metric_rows = []
for raw_concept, metric in detection_metrics.items():
    clean_concept = _clean_concept_name(raw_concept)
    if clean_concept in filtered_clean:
        metric_rows.append((
            clean_concept,
            float(metric.get("F1", 0.0)),
            float(metric.get("accuracy", 0.0)),
        ))

sorted_rows = sorted(metric_rows, key=lambda item: (item[1], item[2]), reverse=True)
sorted_concepts = [(concept, f1) for concept, f1, _acc in sorted_rows]

if not sorted_concepts:
    # Fallback: do not block CT-CBM if all concepts were filtered by a naming mismatch
    # or by an overly strict F1 cutoff. This still orders concepts deterministically.
    sorted_rows = sorted(
        [(_clean_concept_name(c), float(m.get("F1", 0.0)), float(m.get("accuracy", 0.0))) for c, m in detection_metrics.items()],
        key=lambda item: (item[1], item[2]),
        reverse=True,
    )
    sorted_concepts = [(concept, f1) for concept, f1, _acc in sorted_rows]

with open(ranking_dir / "sorted_macro_concepts.json", "w") as f:
    json.dump(dict(sorted_concepts), f, indent=2)

print("Filtered concepts:", len(filtered_concepts))
print("Ranked concepts:", len(sorted_concepts))
print("Top ranked concepts:", sorted_concepts[:10])


In [ ]:
# LIG concept ranking and combined score.
# The paper notebooks combine concept identifiability (CAV detection F1) with
# LIG-based concept importance, then use that combined ranking for coverage.
# If RUN_LIG_RANKING=False, the notebook keeps the faster F1-only ranking from
# the previous cell for quick debugging.

if RUN_LIG_RANKING:
    import os, pickle, gc
    from captum.attr import LayerIntegratedGradients
    from LIG_ranking import compute_cosine_similarities, postprocess_cosine
    if config.model_name == "gemma":
        from LIG_ranking import compute_attributions_on_gemma as compute_attributions
    else:
        from LIG_ranking import compute_attributions

    # Use the fine-tuned black-box model for LIG, matching the original notebooks.
    black_box_model.embedder_model.to(config.device)
    black_box_model.classifier.to(config.device)
    black_box_model.embedder_model.eval()
    black_box_model.classifier.eval()
    embedder_tokenizer.model_max_length = config.max_len

    if config.model_name == "gemma":
        def forward_LIG_black_box(input_ids, attention_mask=None):
            outputs = black_box_model.embedder_model(input_ids=input_ids)
            pooled_output = outputs[0][:, -1, :]
            return black_box_model.classifier(pooled_output)

        lig = LayerIntegratedGradients(
            forward_LIG_black_box,
            layer=black_box_model.embedder_model.layers[-1],
        )
    else:
        def forward_LIG_black_box(input_ids, attention_mask):
            outputs = black_box_model.embedder_model(
                input_ids=input_ids.to(config.device),
                attention_mask=attention_mask.to(config.device),
            )
            if config.use_cls_token:
                pooled_output = outputs.last_hidden_state[:, 0, :]
            else:
                pooled_output = outputs.last_hidden_state.mean(1)
            return black_box_model.classifier(pooled_output)

        lig = LayerIntegratedGradients(
            forward_LIG_black_box,
            layer=black_box_model.embedder_model.encoder.layer[-1],
        )

    df_for_lig = df_aug_train.reset_index(drop=True).copy()
    attr_path = ranking_dir / f"attributions_df_{config.cavs_type}_{config.annotation}.pkl"
    if attr_path.exists():
        with open(attr_path, "rb") as f:
            attributions_df = pickle.load(f)
        if len(attributions_df) != len(df_for_lig):
            attr_path.unlink()
            attributions_df = None
            print("Removed stale LIG attribution cache:", attr_path)
    else:
        attributions_df = None

    if attributions_df is None:
        attributions_df = compute_attributions(
            df_for_lig,
            LIG_ATTRIBUTION_BATCH_SIZE,
            embedder_tokenizer,
            lig,
            config.device,
        )
        with open(attr_path, "wb") as f:
            pickle.dump(attributions_df, f)
        print("Saved LIG attributions to", attr_path)
    else:
        print("Loaded LIG attributions from", attr_path)

    lig_cosine_path = ranking_dir / f"df_aug_train_updated_{config.annotation}.pkl"
    expected_lig_cols = {f"cos_{concept}" for concept in cavs.keys()}
    if lig_cosine_path.exists():
        with open(lig_cosine_path, "rb") as f:
            df_lig_updated = pickle.load(f)
        cache_ok = len(df_lig_updated) == len(df_for_lig) and expected_lig_cols.issubset(set(df_lig_updated.columns))
        if not cache_ok:
            lig_cosine_path.unlink()
            df_lig_updated = None
            print("Removed stale LIG cosine cache:", lig_cosine_path)
    else:
        df_lig_updated = None

    if df_lig_updated is None:
        cavs_for_lig = {
            concept: (value if isinstance(value, torch.Tensor) else torch.tensor(value, dtype=torch.float32)).to(config.device)
            for concept, value in cavs.items()
        }
        df_lig_updated = compute_cosine_similarities(attributions_df, df_for_lig.copy(), cavs_for_lig, config.device)
        with open(lig_cosine_path, "wb") as f:
            pickle.dump(df_lig_updated, f)
        print("Saved LIG cosine dataframe to", lig_cosine_path)
    else:
        print("Loaded LIG cosine dataframe from", lig_cosine_path)

    df_lig_updated, lig_sorted_concepts = postprocess_cosine(
        df_lig_updated,
        list(cavs.keys()),
        mode=config.agg_mode,
        agg_scope=config.agg_scope,
    )
    lig_sorted_path = ranking_dir / f"sorted_macro_concepts_cosine_sm_{config.cavs_type}_{config.annotation}_{config.agg_mode}_{config.agg_scope}.json"
    with open(lig_sorted_path, "w") as f:
        json.dump(lig_sorted_concepts, f, indent=2)

    f1_scores = {_clean_concept_name(concept): float(metric.get("F1", 0.0)) for concept, metric in detection_metrics.items()}
    lig_scores = {_clean_concept_name(concept): float(score) for concept, score in lig_sorted_concepts}
    concepts_for_combination = sorted(set(f1_scores) & set(lig_scores))

    combined_df = pd.DataFrame({
        "concept": concepts_for_combination,
        "F1 Score detected": [f1_scores[c] for c in concepts_for_combination],
        "LIG score": [lig_scores[c] for c in concepts_for_combination],
    })
    if combined_df.empty:
        print("No overlapping concepts between F1 and LIG rankings; keeping F1-only ranking.")
    else:
        lig_min = combined_df["LIG score"].min()
        lig_max = combined_df["LIG score"].max()
        if lig_max > lig_min:
            combined_df["LIG score norm"] = (combined_df["LIG score"] - lig_min) / (lig_max - lig_min)
        else:
            combined_df["LIG score norm"] = 1.0
        combined_df["combined_score_LIG"] = combined_df["F1 Score detected"] * combined_df["LIG score norm"]
        combined_df = combined_df.sort_values("combined_score_LIG", ascending=False)

        combined_csv_path = ranking_dir / f"combined_score_concept_{config.annotation}_{config.agg_mode}_{config.agg_scope}.csv"
        combined_df.to_csv(combined_csv_path, index=False)
        combined_score_LIG = dict(zip(combined_df["concept"], combined_df["combined_score_LIG"]))
        combined_json_path = ranking_dir / f"combined_score_LIG_{config.annotation}_{config.agg_mode}_{config.agg_scope}.json"
        with open(combined_json_path, "w") as f:
            json.dump(combined_score_LIG, f, indent=2)

        sorted_concepts = list(combined_score_LIG.items())
        print("Saved combined LIG ranking to", combined_json_path)
        print("Top combined LIG concepts:", sorted_concepts[:10])

    gc.collect()
    torch.cuda.empty_cache()
else:
    print("Skipping LIG ranking; using F1-only concept ranking for coverage.")


In [ ]:
# Coverage ranking for CT-CBM.
# CT-CBM starts with enough high-ranked concepts to cover a target fraction of examples.
# By default sorted_concepts comes from the combined LIG score above; if LIG is disabled,
# it falls back to the concept identifiability ranking.
# The full_pipeline implementation expects this exact JSON path for strategy='new_heuristique'.

from new_heuristique import clean_concept_name, rename_dataframe_columns


def compute_cumulative_coverage_dict(df, sorted_concepts):
    # Clean columns in the same way as the repo heuristic before checking coverage.
    clean_df = rename_dataframe_columns(df.copy())
    total = max(len(clean_df), 1)
    cumulative_cols = []
    coverage_by_concept = {}

    for concept, _score in sorted_concepts:
        concept = clean_concept_name(concept)
        col = f"dummy_{concept}" if f"dummy_{concept}" in clean_df.columns else concept
        if col not in clean_df.columns:
            continue
        cumulative_cols.append(col)
        covered = clean_df[cumulative_cols].eq(1).any(axis=1).sum()
        coverage_by_concept[concept] = float(covered / total)
    return coverage_by_concept

coverage_ranking = compute_cumulative_coverage_dict(df_aug_train, sorted_concepts)
coverage_path = ranking_dir / f"sorted_macro_concepts_coverage_{config.annotation}_{config.agg_mode}_{config.agg_scope}.json"
with open(coverage_path, "w") as f:
    json.dump(coverage_ranking, f, indent=2)

print("Coverage file:", coverage_path)
print("Top coverage entries:", list(coverage_ranking.items())[:10])

In [ ]:
# Initialize the CT-CBM model.
# The joint model learns X->C concept heads and C->Y task prediction.
# The residual layer captures the remaining task signal; CT-CBM iteratively adds concepts until
# the residual no longer improves validation performance.

import importlib

if config.model_name == "gemma":
    import full_pipeline_gemma as _full_pipeline_module
    import models.jointCBMv2_gemma as _joint_module
else:
    import full_pipeline as _full_pipeline_module
    import models.jointCBMv2 as _joint_module

# Reload local modules so Colab picks up fixes made after an earlier import.
_full_pipeline_module = importlib.reload(_full_pipeline_module)
_joint_module = importlib.reload(_joint_module)
JointResidualFittingModel = _full_pipeline_module.JointResidualFittingModel
CTJointModel = _joint_module.JointModel

# Start with a placeholder bottleneck; full_pipeline reinitializes the bottleneck once concepts are selected.
embedder_model, embedder_tokenizer, ModelXtoCtoY_layer, _ = load_model_and_tokenizer(config, n_concepts=1)
ct_joint = CTJointModel(embedder_model, embedder_tokenizer, ModelXtoCtoY_layer, config, train_aug_loader, val_loader)

linear_layer = RidgeLinearLayer(config.dim, config.num_labels, l2_lambda=config.l2_lambda)
linear_layer.to(config.device)
ct_joint.linear_layer = linear_layer

joint_residual_model = JointResidualFittingModel(
    joint_model=ct_joint,
    linear_layer=linear_layer,
    discovery_model=None,
    discovery_tokenizer=None,
    config=config,
)

print("CT-CBM initialized.")

In [ ]:
# Run CT-CBM.
# This is the main training loop from the paper implementation, excluding comparison methods.
# It uses the coverage ranking built from our annotation and mean CAVs.

if RUN_CT_CBM_TRAINING:
    joint_residual_model.run_full_pipeline_tcavs_strategy(
        train_loader=train_aug_loader,
        val_loader=val_loader,
        test_loader=test_aug_loader,
        num_iterations=NUM_ITERATIONS,
        num_epochs_residual_layer=NUM_EPOCHS_RESIDUAL,
        cavs_type_arg="mean",
        strategy="new_heuristique",
        coverage_threshold=COVERAGE_THRESHOLD,
    )
else:
    print("Skipping CT-CBM training. All pre-training artifacts were prepared.")

In [ ]:
# Inspect saved CT-CBM outputs.
# The final files include model checkpoints, CAVs, detection metrics, coverage ranking, and performance JSON.

for path in sorted(Path(config.SAVE_PATH).rglob("*.json")):
    print(path)
for path in sorted(Path(config.SAVE_PATH).rglob("*.pth")):
    print(path)

# CT-CBM test classification reports

Print classification reports for the CT-CBM simple branch and the CT-CBM residual branch on the held-out test set.


In [ ]:
# CT-CBM test classification reports.
# "Simple" uses the concept bottleneck path X->C->Y only.
# "Residual" adds the fitted residual layer output to the concept bottleneck logits.

from sklearn.metrics import classification_report
import pandas as pd

if "joint_residual_model" not in globals():
    raise RuntimeError("Run the CT-CBM initialization/training cells before this report cell.")

ct_cbm_model = joint_residual_model.joint_model
ct_cbm_model.embedder_model.eval()
ct_cbm_model.ModelXtoCtoY_layer.eval()
if getattr(ct_cbm_model, "linear_layer", None) is not None:
    ct_cbm_model.linear_layer.eval()

y_true = []
y_pred_simple = []
y_pred_residual = []

with torch.no_grad():
    for batch in test_aug_loader:
        labels = batch["label"].to(config.device)

        # Simple CT-CBM branch: use only the X->C->Y logits returned by forward(batch).
        outputs, _ = ct_cbm_model.forward(batch)
        simple_logits = outputs[0]
        simple_preds = torch.argmax(simple_logits, dim=1)

        # Residual CT-CBM branch: add residual logits to the concept bottleneck logits.
        input_ids = batch["input_ids"].to(config.device)
        attention_mask = batch["attention_mask"].to(config.device)
        residual_logits, _, _ = ct_cbm_model.forward_linear(input_ids, attention_mask)
        residual_preds = torch.argmax(residual_logits, dim=1)

        y_true.extend(labels.cpu().tolist())
        y_pred_simple.extend(simple_preds.cpu().tolist())
        y_pred_residual.extend(residual_preds.cpu().tolist())

print("CT-CBM simple classification report")
print(classification_report(y_true, y_pred_simple, digits=4, zero_division=0))

print("CT-CBM residual classification report")
print(classification_report(y_true, y_pred_residual, digits=4, zero_division=0))

ct_cbm_report_df = pd.DataFrame({
    "label": y_true,
    "ct_cbm_simple_pred": y_pred_simple,
    "ct_cbm_residual_pred": y_pred_residual,
})
report_path = Path(config.SAVE_PATH) / "ct_cbm_test_predictions_simple_and_residual.csv"
ct_cbm_report_df.to_csv(report_path, index=False)
print("Saved CT-CBM test predictions to", report_path)
